# Monte Carlo: a miss is a probability

Uses the Python mirror `pymiss` (same formulas and names as `src/physics`). Run from the repo root.


In [ ]:
import sys, math
sys.path.insert(0, '..' if 'notebooks' in __import__('os').getcwd() else '.')
from pymiss.physics.constants import AU, MU_SUN, MU_HOME, R_HOME, YEAR, DAY
from pymiss.physics.close_approach import close_approach, sphere_mass
from pymiss.physics.elements import state_to_elements
from pymiss.physics.kepler import Elements, propagate
from pymiss.physics.kinetic import kinetic_impact
from pymiss.sim.rng import Rng
home = Elements(AU, 0.0, 0.0, 0.0, 0.0)
T_HIT, LEAD, V_INF = 6 * YEAR, 5 * YEAR, 10e3
hx, hy, hvx, hvy = propagate(home, MU_SUN, T_HIT)
r, v = math.hypot(hx, hy), math.hypot(hvx, hvy)
c, s = math.cos(math.pi / 3), math.sin(math.pi / 3)
rock = state_to_elements(hx, hy, hvx + V_INF * (c * hvx / v + s * hx / r), hvy + V_INF * (c * hvy / v + s * hy / r), MU_SUN, T_HIT)
M = sphere_mass(80, 2000)
x, y, vx, vy = propagate(rock, MU_SUN, T_HIT - LEAD)
vm = math.hypot(vx, vy)
rng = Rng(2026)
N, sigma_v, misses = 200, 0.005, 0
for i in range(N):
    beta = rng.float(1, 4)           # same draw order as the lab: beta, then the velocity error
    err = rng.gaussian(0, sigma_v)
    dvx, dvy, U, Q = kinetic_impact(600, M, vx * (1 + 1e4 / vm), vy * (1 + 1e4 / vm), vx, vy, beta)
    el = state_to_elements(x, y, vx + dvx + err * vx / vm, vy + dvy + err * vy / vm, MU_SUN, T_HIT - LEAD)
    ca = close_approach(el, home, MU_SUN, MU_HOME, R_HOME, T_HIT, window=2 * DAY, coarse=4 * 3600, tol=10)
    misses += 0 if ca.impact else 1
p = misses / N
print('P(MISS) = %.3f +/- %.3f' % (p, math.sqrt(p * (1 - p) / N)))